In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ---------------------------------------------------
# CONFIG
# ---------------------------------------------------

np.random.seed(42)

N = 50_000

# ---------------------------------------------------
# HELPERS
# ---------------------------------------------------

def random_date(start, end, size):
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    days = (end - start).days

    return start + pd.to_timedelta(
        np.random.randint(0, days + 1, size=size),
        unit="D"
    )


def messy_date(date_value):
    """
    Convert a valid datetime into a randomly selected messy representation.
    """

    if pd.isna(date_value):
        return np.random.choice(
            ["", "NA", "N/A", "NULL", "None"],
            p=[0.35, 0.15, 0.20, 0.20, 0.10]
        )

    formats = [
        "%Y-%m-%d",
        "%m/%d/%Y",
        "%m-%d-%Y",
        "%b %d %Y",
        "%Y%m%d",
        "%m/%d/%y"
    ]

    fmt = np.random.choice(formats)

    return pd.Timestamp(date_value).strftime(fmt)


def messy_percent(value):
    """
    Produce percentages in inconsistent formats.
    """

    choices = np.random.randint(0, 6)

    if choices == 0:
        return round(value, 2)

    elif choices == 1:
        return f"{value:.2f}%"

    elif choices == 2:
        return round(value / 100, 4)

    elif choices == 3:
        return f" {value:.1f} "

    elif choices == 4:
        return f"{value:.3f}%"

    else:
        return str(round(value, 2))


def corrupt_category(value, mapping):
    """
    Replace a clean categorical value with a messy version.
    """

    options = mapping.get(value, [value])

    return np.random.choice(options)


# ---------------------------------------------------
# BASE DATA
# ---------------------------------------------------

df = pd.DataFrame()

# Loan number
df["loan_number"] = np.arange(10000001, 10000001 + N).astype(str)

# Application dates
df["application_date_clean"] = random_date(
    "2023-01-01",
    "2026-06-30",
    N
)

# Lock date typically 1-45 days after application
lock_delay = np.random.randint(1, 46, size=N)

df["lock_date_clean"] = (
    df["application_date_clean"]
    + pd.to_timedelta(lock_delay, unit="D")
)

# ---------------------------------------------------
# CREDIT / LOAN CHARACTERISTICS
# ---------------------------------------------------

df["FICO_clean"] = np.clip(
    np.random.normal(730, 55, N),
    500,
    850
).round().astype(int)

df["LTV_clean"] = np.clip(
    np.random.normal(76, 15, N),
    20,
    105
)

# CLTV usually >= LTV
df["CLTV_clean"] = np.clip(
    df["LTV_clean"] + np.random.normal(3, 6, N),
    20,
    115
)

df["DTI_clean"] = np.clip(
    np.random.normal(37, 10, N),
    5,
    65
)

df["note_rate_clean"] = np.clip(
    np.random.normal(6.25, 1.0, N),
    2.0,
    9.5
)

# ---------------------------------------------------
# UNDERWRITING SYSTEM
# ---------------------------------------------------

uw_choices = ["DU", "LPA", "Manual"]

df["UW_system_clean"] = np.random.choice(
    uw_choices,
    size=N,
    p=[0.58, 0.32, 0.10]
)

# ---------------------------------------------------
# LOAN PROGRAM
# ---------------------------------------------------

program_choices = [
    "Conventional 30YR",
    "Conventional 15YR",
    "FHA 30YR",
    "VA 30YR",
    "Jumbo 30YR"
]

df["loan_program_clean"] = np.random.choice(
    program_choices,
    size=N,
    p=[0.55, 0.10, 0.15, 0.10, 0.10]
)

# ---------------------------------------------------
# FUNDING PROBABILITY
# ---------------------------------------------------

# Start around 75% pull-through
log_odds = np.full(N, 1.10)

# Higher FICO improves probability
log_odds += (df["FICO_clean"] - 700) * 0.004

# Higher DTI hurts probability
log_odds -= (df["DTI_clean"] - 35) * 0.025

# High LTV slightly hurts
log_odds -= np.maximum(df["LTV_clean"] - 80, 0) * 0.015

# UW impacts
log_odds += np.where(
    df["UW_system_clean"] == "DU",
    0.20,
    0
)

log_odds += np.where(
    df["UW_system_clean"] == "LPA",
    0.15,
    0
)

log_odds -= np.where(
    df["UW_system_clean"] == "Manual",
    0.30,
    0
)

# Program impacts
log_odds -= np.where(
    df["loan_program_clean"] == "Jumbo 30YR",
    0.20,
    0
)

log_odds += np.where(
    df["loan_program_clean"] == "VA 30YR",
    0.10,
    0
)

fund_prob = 1 / (1 + np.exp(-log_odds))

df["funded_flag"] = np.random.binomial(
    1,
    fund_prob
)

# ---------------------------------------------------
# FINAL STATUS
# ---------------------------------------------------

fallout_statuses = [
    "Withdrawn",
    "Denied",
    "Cancelled",
    "Expired"
]

df["file_status_clean"] = np.where(
    df["funded_flag"] == 1,
    "Funded",
    np.random.choice(
        fallout_statuses,
        size=N,
        p=[0.40, 0.25, 0.25, 0.10]
    )
)

# ---------------------------------------------------
# FUNDING DATE
# ---------------------------------------------------

fund_delay = np.random.randint(
    10,
    91,
    size=N
)

df["funded_date_clean"] = pd.NaT

funded_mask = df["funded_flag"] == 1

df.loc[
    funded_mask,
    "funded_date_clean"
] = (
    df.loc[funded_mask, "lock_date_clean"]
    + pd.to_timedelta(
        fund_delay[funded_mask],
        unit="D"
    )
)

# ---------------------------------------------------
# ACTION DATE
# ---------------------------------------------------

action_delay = np.random.randint(
    3,
    75,
    size=N
)

df["action_date_clean"] = pd.NaT

fallout_mask = df["funded_flag"] == 0

df.loc[
    fallout_mask,
    "action_date_clean"
] = (
    df.loc[fallout_mask, "lock_date_clean"]
    + pd.to_timedelta(
        action_delay[fallout_mask],
        unit="D"
    )
)

# ---------------------------------------------------
# CREATE MESSY DATE COLUMNS
# ---------------------------------------------------

df["application_date"] = df[
    "application_date_clean"
].apply(messy_date)

df["lock_date"] = df[
    "lock_date_clean"
].apply(messy_date)

df["funded_date"] = df[
    "funded_date_clean"
].apply(messy_date)

df["action_date"] = df[
    "action_date_clean"
].apply(messy_date)

# ---------------------------------------------------
# CREATE MESSY NUMERIC FIELDS
# ---------------------------------------------------

df["note_rate"] = df[
    "note_rate_clean"
].apply(messy_percent)

df["LTV"] = df[
    "LTV_clean"
].apply(messy_percent)

df["CLTV"] = df[
    "CLTV_clean"
].apply(messy_percent)

df["DTI"] = df[
    "DTI_clean"
].apply(messy_percent)

df["FICO"] = df[
    "FICO_clean"
].astype(object)

# ---------------------------------------------------
# MESSY UW LABELS
# ---------------------------------------------------

uw_messy_map = {
    "DU": [
        "DU",
        "du",
        "D.U.",
        "Desktop Underwriter",
        "DESKTOP UNDERWRITER",
        " DU "
    ],

    "LPA": [
        "LPA",
        "lpa",
        "LP",
        "Loan Prospector",
        "Loan Product Advisor",
        " LPA "
    ],

    "Manual": [
        "Manual",
        "manual",
        "MANUAL UW",
        "Man UW",
        "manual underwriting"
    ]
}

df["UW_system"] = df[
    "UW_system_clean"
].apply(
    lambda x: corrupt_category(
        x,
        uw_messy_map
    )
)

# ---------------------------------------------------
# MESSY FILE STATUS
# ---------------------------------------------------

status_messy_map = {
    "Funded": [
        "Funded",
        "funded",
        "FUNDED",
        "Fund",
        "Closed/Funded",
        " Funded "
    ],

    "Withdrawn": [
        "Withdrawn",
        "withdrawn",
        "W/D",
        "WD",
        "Withdraw",
        " WITHDRAWN "
    ],

    "Denied": [
        "Denied",
        "denied",
        "DENY",
        "Declined",
        "Credit Denied"
    ],

    "Cancelled": [
        "Cancelled",
        "Canceled",
        "CXL",
        "cancel",
        "CANCELLED"
    ],

    "Expired": [
        "Expired",
        "expired",
        "EXP",
        "Lock Expired"
    ]
}

df["file_status"] = df[
    "file_status_clean"
].apply(
    lambda x: corrupt_category(
        x,
        status_messy_map
    )
)

# ---------------------------------------------------
# MESSY LOAN PROGRAM
# ---------------------------------------------------

program_messy_map = {

    "Conventional 30YR": [
        "Conventional 30YR",
        "Conv 30",
        "CONV30",
        "30yr fixed",
        "30 Year Conventional",
        "Conv 30yr",
        " conventional 30 "
    ],

    "Conventional 15YR": [
        "Conventional 15YR",
        "Conv 15",
        "15yr fixed",
        "CONV15",
        "15 Year Conventional"
    ],

    "FHA 30YR": [
        "FHA 30YR",
        "FHA",
        "F.H.A.",
        "FHA30",
        "30YR FHA"
    ],

    "VA 30YR": [
        "VA 30YR",
        "VA",
        "V.A.",
        "VA30",
        "30 Year VA"
    ],

    "Jumbo 30YR": [
        "Jumbo 30YR",
        "Jumbo",
        "JUMBO30",
        "30yr jumbo",
        "Non-Conforming 30"
    ]
}

df["loan_program"] = df[
    "loan_program_clean"
].apply(
    lambda x: corrupt_category(
        x,
        program_messy_map
    )
)

# ---------------------------------------------------
# INJECT DATA QUALITY ISSUES
# ---------------------------------------------------

# Duplicate some loan numbers
duplicate_idx = np.random.choice(
    df.index,
    size=500,
    replace=False
)

source_idx = np.random.choice(
    df.index,
    size=500,
    replace=True
)

df.loc[
    duplicate_idx,
    "loan_number"
] = df.loc[
    source_idx,
    "loan_number"
].values


# Blank/null loan numbers
bad_loan_idx = np.random.choice(
    df.index,
    size=150,
    replace=False
)

df.loc[
    bad_loan_idx,
    "loan_number"
] = np.random.choice(
    ["", "NA", "NULL", "None"],
    size=len(bad_loan_idx)
)


# Bad FICO values
fico_bad_idx = np.random.choice(
    df.index,
    size=700,
    replace=False
)

df.loc[
    fico_bad_idx,
    "FICO"
] = np.random.choice(
    [
        "N/A",
        "NULL",
        "",
        999,
        420,
        " 740 ",
        "seven hundred"
    ],
    size=len(fico_bad_idx)
)


# Bad LTV values
ltv_bad_idx = np.random.choice(
    df.index,
    size=600,
    replace=False
)

df.loc[
    ltv_bad_idx,
    "LTV"
] = np.random.choice(
    [
        "150%",
        "-20%",
        "999",
        "N/A",
        "",
        "1O5%"
    ],
    size=len(ltv_bad_idx)
)


# Bad CLTV values
cltv_bad_idx = np.random.choice(
    df.index,
    size=500,
    replace=False
)

df.loc[
    cltv_bad_idx,
    "CLTV"
] = np.random.choice(
    [
        "180%",
        "-5",
        "N/A",
        "NULL",
        "",
        "9O%"
    ],
    size=len(cltv_bad_idx)
)


# Bad DTI
dti_bad_idx = np.random.choice(
    df.index,
    size=500,
    replace=False
)

df.loc[
    dti_bad_idx,
    "DTI"
] = np.random.choice(
    [
        "-10%",
        "110%",
        "N/A",
        "",
        "NULL",
        "forty five"
    ],
    size=len(dti_bad_idx)
)


# Bad note rates
rate_bad_idx = np.random.choice(
    df.index,
    size=500,
    replace=False
)

df.loc[
    rate_bad_idx,
    "note_rate"
] = np.random.choice(
    [
        "65%",
        "-3%",
        "N/A",
        "",
        "NULL",
        "six point five"
    ],
    size=len(rate_bad_idx)
)


# ---------------------------------------------------
# DATE CORRUPTION
# ---------------------------------------------------

# Lock date before application
bad_lock_idx = np.random.choice(
    df.index,
    size=400,
    replace=False
)

bad_lock_dates = (
    df.loc[
        bad_lock_idx,
        "application_date_clean"
    ]
    - pd.to_timedelta(
        np.random.randint(
            1,
            30,
            size=len(bad_lock_idx)
        ),
        unit="D"
    )
)

df.loc[
    bad_lock_idx,
    "lock_date"
] = bad_lock_dates.apply(messy_date)


# Funded dates before lock dates
bad_fund_idx = np.random.choice(
    df.index[df["funded_flag"] == 1],
    size=350,
    replace=False
)

bad_fund_dates = (
    df.loc[
        bad_fund_idx,
        "lock_date_clean"
    ]
    - pd.to_timedelta(
        np.random.randint(
            1,
            20,
            size=len(bad_fund_idx)
        ),
        unit="D"
    )
)

df.loc[
    bad_fund_idx,
    "funded_date"
] = bad_fund_dates.apply(messy_date)


# Impossible dates
impossible_idx = np.random.choice(
    df.index,
    size=300,
    replace=False
)

df.loc[
    impossible_idx,
    "application_date"
] = np.random.choice(
    [
        "02/30/2025",
        "13/15/2024",
        "2025-99-01",
        "00/12/2026",
        "31/31/2025"
    ],
    size=len(impossible_idx)
)


# ---------------------------------------------------
# RANDOM NULL MARKERS
# ---------------------------------------------------

messy_columns = [
    "application_date",
    "lock_date",
    "note_rate",
    "LTV",
    "CLTV",
    "DTI",
    "FICO",
    "UW_system",
    "funded_date",
    "action_date",
    "file_status",
    "loan_program"
]

for col in messy_columns:

    null_idx = np.random.choice(
        df.index,
        size=int(N * 0.01),
        replace=False
    )

    df.loc[
        null_idx,
        col
    ] = np.random.choice(
        [
            "",
            "NA",
            "N/A",
            "NULL",
            "None"
        ],
        size=len(null_idx)
    )


# ---------------------------------------------------
# FINAL PUBLIC DATASET
# ---------------------------------------------------

final_columns = [
    "loan_number",
    "application_date",
    "lock_date",
    "note_rate",
    "LTV",
    "CLTV",
    "DTI",
    "FICO",
    "UW_system",
    "funded_date",
    "action_date",
    "file_status",
    "loan_program"
]

df_final = df[final_columns].copy()

# Shuffle rows
df_final = df_final.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# ---------------------------------------------------
# EXPORT
# ---------------------------------------------------

df_final.to_csv(
    "mortgage_pull_through_messy.csv",
    index=False
)

print(df_final.head(20))

print("\nShape:")
print(df_final.shape)

print("\nData Types:")
print(df_final.dtypes)

print("\nSaved:")
print("mortgage_pull_through_messy.csv")

   loan_number application_date    lock_date note_rate      LTV     CLTV  \
0     10033554       05/21/2026     06/09/26      4.03   0.7864    73.5    
1     10009428       08/05/2024  Sep 10 2024      7.7    0.5081   0.5349   
2     10000200         07/06/23  Jul 14 2023      6.73  71.874%  83.581%   
3     10012448       2023-02-11     20230212    6.670%   83.85%    87.5    
4     10039490         02/19/24  Mar 23 2024      4.74   0.7702    72.93   
5     10042725       01-17-2024     20240221     6.09%   0.8427    84.27   
6     10010823         20230527   06/15/2023    0.0662    82.56    87.2    
7     10049499       12-22-2024     20241226    0.0648    92.0   91.691%   
8     10004145       06/11/2024   2024-07-16      7.48  85.877%  85.635%   
9     10036959       08/17/2023   2023-09-14      6.64    64.73   54.13%   
10    10043107       2024-07-17   08/09/2024    0.0587    75.5     72.1    
11    10038696         05/01/23   05-04-2023    6.438%    74.21   80.12%   
12    100061